In [243]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [244]:
pathAcceptedLoan = "../data/raw/accepted_2007_to_2018Q4.csv"
pathRejectedLoan = "../data/raw/rejected_2007_to_2018Q4.csv"

In [245]:
AcceptedLoan_df = pd.read_csv(pathAcceptedLoan, low_memory=False)

In [246]:
AcceptedLoan_df['term'] = AcceptedLoan_df['term'].str.extract(r'(\d+)')
AcceptedLoan_df['term'] = pd.to_numeric(AcceptedLoan_df['term'], errors='coerce')
AcceptedLoan_df.replace([' ', '', 'NA', 'N/A', 'null', 'None'], np.nan, inplace=True)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36.0,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36.0,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60.0,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60.0,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60.0,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260696,88985880,NaN,40000.0,40000.0,40000.0,60.0,10.49,859.56,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260697,88224441,NaN,24000.0,24000.0,24000.0,60.0,14.49,564.56,C,C4,...,NaN,NaN,Cash,Y,Mar-2019,ACTIVE,Mar-2019,10000.0,44.82,1.0
2260698,88215728,NaN,14000.0,14000.0,14000.0,60.0,14.49,329.33,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260699,Total amount funded in policy code 1: 1465324575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:

AcceptedLoan_df['issue_d'] = pd.to_datetime(
    AcceptedLoan_df['issue_d'], 
    format='%b-%Y',
    errors='coerce'
)

In [ ]:
for col in AcceptedLoan_df.columns:
    try:
        AcceptedLoan_df[col] = pd.to_numeric(AcceptedLoan_df[col])
    except:
        pass


column_drop = []
size = AcceptedLoan_df.shape[0]

for col in AcceptedLoan_df.columns:
    series = AcceptedLoan_df[col]
    null_ratio = series.isnull().sum() / size * 100
  
    if null_ratio >= 30:
        column_drop.append(col)
        continue

    if null_ratio > 0:
        numeric_ratio = pd.to_numeric(AcceptedLoan_df[col], errors='coerce').notnull().sum() / size * 100

        if numeric_ratio > 0:
            median_value = pd.to_numeric(AcceptedLoan_df[col], errors='coerce').median()
            AcceptedLoan_df[col] = pd.to_numeric(AcceptedLoan_df[col], errors='coerce').fillna(median_value)
        else:
            mode_value = series.mode()
            if not mode_value.empty:
                AcceptedLoan_df[col] = series.fillna(mode_value.iloc[0])
            else:
                column_drop.append(col)

print("Columns to drop:", column_drop)
AcceptedLoan_df.drop(columns=column_drop, inplace=True)
AcceptedLoan_df.shape

In [ ]:
AcceptedLoan_df.drop(columns=['id'], inplace=True)

In [ ]:
AcceptedLoan_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 92 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   loan_amnt                   float64
 1   funded_amnt                 float64
 2   funded_amnt_inv             float64
 3   term                        float64
 4   int_rate                    float64
 5   installment                 float64
 6   grade                       int8   
 7   sub_grade                   int8   
 8   emp_title                   float64
 9   emp_length                  int8   
 10  home_ownership              int8   
 11  annual_inc                  float64
 12  verification_status         int8   
 13  issue_d                     int16  
 14  loan_status                 int8   
 15  pymnt_plan                  str    
 16  url                         str    
 17  purpose                     int8   
 18  title                       float64
 19  zip_code                    str 

In [ ]:
cat_cols = AcceptedLoan_df.select_dtypes(include=['object', 'category', 'string']).columns
for col in cat_cols:
    print(f'{col} | {AcceptedLoan_df[col].dtype} | {AcceptedLoan_df[col].value_counts()} \n{AcceptedLoan_df[col].unique()}\n\n')


pymnt_plan | str | pymnt_plan
n    2260081
y        620
Name: count, dtype: int64 
<StringArray>
['n', 'y']
Length: 2, dtype: str


url | str | url
https://lendingclub.com/browse/loanDetail.action?loan_id=1000007     34
https://lendingclub.com/browse/loanDetail.action?loan_id=68407277     1
https://lendingclub.com/browse/loanDetail.action?loan_id=68355089     1
https://lendingclub.com/browse/loanDetail.action?loan_id=68341763     1
https://lendingclub.com/browse/loanDetail.action?loan_id=66310712     1
                                                                     ..
https://lendingclub.com/browse/loanDetail.action?loan_id=89885898     1
https://lendingclub.com/browse/loanDetail.action?loan_id=88977788     1
https://lendingclub.com/browse/loanDetail.action?loan_id=88985880     1
https://lendingclub.com/browse/loanDetail.action?loan_id=88224441     1
https://lendingclub.com/browse/loanDetail.action?loan_id=88215728     1
Name: count, Length: 2260668, dtype: int64 
<StringArray>
['